# 15 高斯混合模型 Gaussian Mixture Model

GMM 假设数据来自多个高斯分布的混合。相比 K-Means 的硬分配，GMM 会给出“每个点属于每个簇的概率”。


## 0. 学习目标和阅读地图

GMM 是理解“软聚类”和 EM 算法的好模型。你需要掌握：

1. 数据来自多个高斯分布混合是什么意思。
2. responsibility 为什么是软标签。
3. EM 的 E step 和 M step 分别在做什么。
4. GMM 和 K-Means 的关系与差异。


## 1. 数学逻辑

混合模型的概率密度：

$$p(x)=\sum_{k=1}^{K}\pi_k \mathcal{N}(x|\mu_k,\Sigma_k)$$

其中 `pi_k` 是第 k 个高斯成分的权重。

EM 算法两步循环：

E step：计算责任度 responsibility：

$$r_{ik}=\frac{\pi_k\mathcal{N}(x_i|\mu_k,\Sigma_k)}{\sum_j\pi_j\mathcal{N}(x_i|\mu_j,\Sigma_j)}$$

M step：用责任度加权更新参数。


## 1.1 推导拆开看：responsibility 是后验概率

对第 i 个样本和第 k 个高斯成分，responsibility 是：

$$r_{ik}=P(z_i=k|x_i)$$

它表示“第 k 个高斯对这个样本负责多少”。

E step 根据当前参数计算 `r_ik`；M step 用 `r_ik` 作为权重更新均值、协方差和混合权重。

这叫 EM：Expectation-Maximization，先估计隐变量分布，再最大化参数。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs

np.random.seed(42)
X, _ = make_blobs(n_samples=260, centers=[[-2, 0], [1.5, 1.0], [2, -1.5]], cluster_std=[0.5, 0.8, 0.35], random_state=42)


## 1.2 GMM 为什么比 K-Means 更柔软

K-Means 会说：这个点属于第 2 簇。GMM 会说：这个点有 70% 属于第 2 个成分，30% 属于第 1 个成分。

这种软分配对边界附近样本更自然，也能表达椭圆形簇。


In [ ]:
# 从零实现：一维 GMM 的 EM，方便看清责任度
x = np.r_[np.random.normal(-2, 0.5, 100), np.random.normal(2, 0.8, 120)]
K = 2
pi = np.ones(K) / K
mu = np.array([-1.0, 1.0])
sigma = np.array([1.0, 1.0])

def normal_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))

for step in range(20):
    resp = np.column_stack([pi[k] * normal_pdf(x, mu[k], sigma[k]) for k in range(K)])
    resp = resp / resp.sum(axis=1, keepdims=True)
    Nk = resp.sum(axis=0)
    pi = Nk / len(x)
    mu = (resp * x[:, None]).sum(axis=0) / Nk
    sigma = np.sqrt((resp * (x[:, None] - mu) ** 2).sum(axis=0) / Nk)

print('pi:', np.round(pi, 3))
print('mu:', np.round(mu, 3))
print('sigma:', np.round(sigma, 3))


## 1.3 从零实现代码怎么读

一维 EM 示例里：

1. `normal_pdf` 计算每个高斯成分的密度。
2. `resp` 归一化后得到责任度。
3. `Nk` 是每个成分的有效样本数。
4. `mu` 和 `sigma` 用责任度加权更新。

二维完整 GMM 只是把方差扩展成协方差矩阵。


In [ ]:
model = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
labels = model.fit_predict(X)
proba = model.predict_proba(X)
print('每个成分权重:', np.round(model.weights_, 3))
print('前 3 个点的簇概率:')
print(np.round(proba[:3], 3))

plt.scatter(X[:,0], X[:,1], c=labels, cmap='tab10', s=24)
plt.scatter(model.means_[:,0], model.means_[:,1], c='red', marker='x', s=120)
plt.title('GaussianMixture 软聚类')
plt.show()


In [ ]:
# 诊断：用 BIC/AIC 辅助选择成分数量
components = range(1, 7)
bics, aics = [], []
for k in components:
    m_gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42)
    m_gmm.fit(X)
    bics.append(m_gmm.bic(X))
    aics.append(m_gmm.aic(X))

plt.plot(list(components), bics, marker='o', label='BIC')
plt.plot(list(components), aics, marker='o', label='AIC')
plt.title('GMM 成分数量选择')
plt.xlabel('n_components')
plt.ylabel('criterion')
plt.legend()
plt.show()


## 2.1 如何诊断 GMM

GMM 常用 AIC/BIC 选择成分数。它们都会惩罚模型复杂度，避免成分越多似然越高导致过拟合。

BIC 通常惩罚更强，更偏向简单模型；AIC 相对更愿意保留复杂模型。


## 2. 常见误区

- GMM 假设每个簇近似高斯形状，不适合任意形状簇。
- EM 可能陷入局部最优，初始化会影响结果。
- 协方差类型越灵活，模型越容易过拟合。

## 3. 小实验

- 改 `n_components`。
- 改 `covariance_type` 为 `diag` 或 `spherical`。
- 观察边界附近样本的概率分布。


## 5. 复习清单

- GMM 是概率模型，不只是聚类算法。
- responsibility 是每个样本对每个成分的软归属概率。
- EM 交替估计隐变量和更新参数。
- GMM 适合椭圆形簇，但不适合任意形状簇。
